# 07_silver_redext_redcos_validation.ipynb — Boyas REDEXT/REDCOS Bronze → Silver Validation

Este notebook procesa los CSV de **Puertos del Estado / REDEXT REDCOS**.

A diferencia de SIMAR, REDMAR o ERA5, esta fuente se usa como **validación observada de oleaje**, no como entrenamiento principal.

Salida principal:

```text
silver/ocean_validation/source=REDEXT_REDCOS/...
```

Columnas principales:

```text
timestamp, station_id, station_name, zona_id, lat, lon, source,
hs, hmax, tp, tm02, wave_direction,
swell_height, swell_period, swell_direction,
wind_wave_height, wind_wave_period,
raw_observations_in_hour, validation_role
```

Notas:
- Se conservan los datos observados como fuente de validación.
- Se resamplea a frecuencia horaria sin imputar huecos.
- Se generan flags de calidad.
- Si no hay coordenadas en el CSV, el notebook no se rompe: deja `zona_id = CAN_VALIDATION_UNASSIGNED` y lo documenta en los reportes.

## Celda 0 — Montar Google Drive

In [14]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [15]:
!pip -q install geopandas pyarrow shapely fiona tqdm scipy

## Celda 2 — Imports, rutas y configuración

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import re
import unicodedata
import json
import shutil
import gc
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data/bronze"
SILVER_DIR = BASE_DIR / "silver"

REDEXT_DIR = BRONZE_DIR / "Puertos del Estado" / "REDEXT REDCOS"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_VALIDATION_DIR = SILVER_DIR / "ocean_validation"
QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

for d in [OUT_VALIDATION_DIR, QC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_NAME = "REDEXT_REDCOS"

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

MIN_VALID_TS = pd.Timestamp("1999-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2031-01-01", tz="UTC")

# Para pruebas rápidas. Deja None para todo.
MAX_FILES_FOR_TEST = None

print("REDEXT_DIR existe:", REDEXT_DIR.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not REDEXT_DIR.exists():
    raise FileNotFoundError(f"No existe REDEXT_DIR: {REDEXT_DIR}")

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError("No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb.")

REDEXT_DIR existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [17]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value.upper()


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def parse_coordinate(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        val = float(value)
        if abs(val) > 180:
            return np.nan
        return val

    s = str(value).strip().upper()
    s = s.replace(",", ".")

    if s in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    sign = 1
    if any(h in s for h in ["W", "O", "S"]):
        sign = -1
    if s.startswith("-"):
        sign = -1

    nums = re.findall(r"-?\d+(?:\.\d+)?", s)

    if not nums:
        return np.nan

    try:
        if len(nums) >= 3 and ("º" in s or "°" in s or "'" in s or '"' in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            seconds = float(nums[2])
            val = deg + minutes / 60 + seconds / 3600
        elif len(nums) >= 2 and ("º" in s or "°" in s or "'" in s):
            deg = abs(float(nums[0]))
            minutes = float(nums[1])
            val = deg + minutes / 60
        else:
            val = abs(float(nums[0])) if sign == -1 else float(nums[0])

        val = sign * abs(val) if sign == -1 else val

        if abs(val) > 180:
            return np.nan

        return val

    except Exception:
        return np.nan


def to_numeric_series(series):
    if series is None:
        return pd.Series(dtype="float64")

    s = series.astype(str).str.strip()

    missing_tokens = {
        "",
        "NA",
        "N/A",
        "NAN",
        "NULL",
        "NONE",
        "-",
        "--",
        "---",
        "S/D",
        "SD",
        "NODATA",
        "NO_DATA",
    }

    s = s.mask(s.str.upper().isin(missing_tokens))
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9eE+\-.]", "", regex=True)

    out = pd.to_numeric(s, errors="coerce")

    # Códigos habituales de missing.
    out = out.mask(out.isin([-99999, -9999, -999, 999, 9999, 99999]))
    out = out.mask(out <= -999)

    return out


def infer_column(df, candidates):
    cols_norm = {normalize_col(col): col for col in df.columns}

    for candidate in candidates:
        candidate_norm = normalize_col(candidate)

        for col_norm, original_col in cols_norm.items():
            if candidate_norm == col_norm:
                return original_col

        for col_norm, original_col in cols_norm.items():
            if candidate_norm in col_norm:
                return original_col

    return None


def find_col_by_patterns(df, patterns, exclude_patterns=None, used_cols=None):
    if exclude_patterns is None:
        exclude_patterns = []
    if used_cols is None:
        used_cols = set()

    for col in df.columns:
        if col in used_cols:
            continue

        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in exclude_patterns):
            continue

        if any(re.search(pat, norm) for pat in patterns):
            return col

    return None


def ensure_utc(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def looks_like_date_string(value):
    s = str(value)
    return bool(
        re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s)
        or re.search(r"\d{4}[/-]\d{1,2}[/-]\d{1,2}", s)
        or re.search(r"\d{8,14}", s)
    )


def parse_compact_datetime_series(series):
    raw = series.astype(str).str.strip()
    raw = raw.str.replace(r"\.0$", "", regex=True)
    digits = raw.str.replace(r"\D", "", regex=True)

    candidates = [
        (14, "%Y%m%d%H%M%S"),
        (12, "%Y%m%d%H%M"),
        (10, "%Y%m%d%H"),
        (8, "%Y%m%d"),
    ]

    best_parsed = None
    best_ratio = 0

    for length, fmt in candidates:
        mask = digits.str.len() == length

        if mask.mean() < 0.5:
            continue

        parsed = pd.to_datetime(
            digits.where(mask),
            format=fmt,
            errors="coerce",
            utc=True,
        )

        ratio = parsed.notna().mean()

        if ratio > best_ratio:
            best_ratio = ratio
            best_parsed = parsed

    if best_parsed is not None and best_ratio >= 0.5:
        ts_valid = best_parsed.dropna()
        if len(ts_valid) and ts_valid.between(MIN_VALID_TS, MAX_VALID_TS).mean() >= 0.8:
            return best_parsed

    return None


def remove_existing_source_partition(base_dir, source_name=SOURCE_NAME):
    source_path = base_dir / f"source={source_name}"
    if source_path.exists():
        shutil.rmtree(source_path)
        print("Eliminada partición antigua:", source_path)


def write_partitioned_parquet(df, base_dir):
    if df.empty:
        print("DataFrame vacío. No se guarda.")
        return

    df = df.copy()
    df["source"] = df["source"].fillna(SOURCE_NAME).astype(str)
    df["isla"] = df["isla"].fillna("ISLA_DESCONOCIDA").astype(str)
    df["year"] = df["year"].astype("int64")

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(base_dir),
        partition_cols=["source", "year", "isla"],
        compression="snappy",
    )


def dataset_count_and_sample(path, source_name=SOURCE_NAME, sample_n=5):
    if not path.exists():
        return 0, pd.DataFrame()

    dataset = ds.dataset(str(path), format="parquet", partitioning="hive")
    count = dataset.count_rows(filter=(ds.field("source") == source_name))

    if count == 0:
        return 0, pd.DataFrame()

    sample = dataset.head(sample_n, filter=(ds.field("source") == source_name)).to_pandas()

    return count, sample

## Celda 4 — Cargar `beach_geography`

In [18]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_zone_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing_zone_cols = [c for c in required_zone_cols if c not in beach_geography.columns]

if missing_zone_cols:
    raise ValueError(f"Faltan columnas en beach_geography: {missing_zone_cols}")

print("beach_geography shape:", beach_geography.shape)
display(beach_geography.head())

gdf_zones = gpd.GeoDataFrame(
    beach_geography.copy(),
    geometry=gpd.points_from_xy(beach_geography["lon"], beach_geography["lat"]),
    crs="EPSG:4326",
)

gdf_zones_m = gdf_zones.to_crs("EPSG:3857")

beach_geography shape: (561, 17)


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


## Celda 5 — Localizar archivos REDEXT/REDCOS

In [19]:
redext_files = sorted(REDEXT_DIR.glob("*.csv"))

if MAX_FILES_FOR_TEST is not None:
    redext_files = redext_files[:MAX_FILES_FOR_TEST]

if not redext_files:
    raise FileNotFoundError(f"No se encontraron CSV en {REDEXT_DIR}")

REDEXT_FILENAME_RE = re.compile(
    r"^(?P<request_id>\d+)_(?P<download_id>\d+)_(?P<station_id>\d+)_(?P<variable_group>[A-Z]+(?:_[A-Z]+)*)_(?P<start>\d{14})_(?P<end>\d{14})\.csv$",
    re.IGNORECASE,
)


def parse_redext_filename(path):
    m = REDEXT_FILENAME_RE.match(path.name)

    if not m:
        return {
            "filename": path.name,
            "request_id": None,
            "download_id": None,
            "station_id": Path(path).stem,
            "variable_group": "UNKNOWN",
            "file_start": pd.NaT,
            "file_end": pd.NaT,
            "filename_parse_ok": False,
        }

    d = m.groupdict()

    return {
        "filename": path.name,
        "request_id": d["request_id"],
        "download_id": d["download_id"],
        "station_id": str(d["station_id"]),
        "variable_group": d["variable_group"].upper(),
        "file_start": pd.to_datetime(d["start"], format="%Y%m%d%H%M%S", errors="coerce", utc=True),
        "file_end": pd.to_datetime(d["end"], format="%Y%m%d%H%M%S", errors="coerce", utc=True),
        "filename_parse_ok": True,
    }


files_df = pd.DataFrame([parse_redext_filename(p) | {"path": str(p)} for p in redext_files])

print("Archivos REDEXT/REDCOS encontrados:", len(files_df))
display(files_df)
display(files_df["variable_group"].value_counts().reset_index())

Archivos REDEXT/REDCOS encontrados: 5


,filename,request_id,download_id,station_id,variable_group,file_start,file_end,filename_parse_ok,path
0,25411_52356_2442_ALL_20010101184619_2026050617...,25411,52356,2442,ALL,2001-01-01 18:46:19+00:00,2026-05-06 17:46:19+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
1,25411_52357_1440_ALL_20070620220000_2014012423...,25411,52357,1440,ALL,2007-06-20 22:00:00+00:00,2014-01-24 23:00:00+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
2,25411_52358_2446_ALL_20010101184650_2026050617...,25411,52358,2446,ALL,2001-01-01 18:46:50+00:00,2026-05-06 17:46:50+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
3,25411_52359_1414_ALL_20010101184657_2026050617...,25411,52359,1414,ALL,2001-01-01 18:46:57+00:00,2026-05-06 17:46:57+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...
4,25411_52360_1421_ALL_20250506174709_2026050617...,25411,52360,1421,ALL,2025-05-06 17:47:09+00:00,2026-05-06 17:47:09+00:00,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...


,variable_group,count
0,ALL,5


## Celda 6 — Fallback opcional de coordenadas de boyas

In [20]:
# El notebook prioriza coordenadas extraídas de la cabecera del CSV.
# Este diccionario queda preparado por si alguna boya no trae lat/lon.
# Por defecto no se fuerza ningún valor inventado.
# Si al ejecutar ves coordenadas faltantes, puedes rellenar manualmente station_id -> lat/lon
# y reejecutar desde la celda 9.

REDEXT_REDCOS_COORD_FALLBACK = {
    # "2442": {"station_name": "Nombre boya", "lat": 28.0, "lon": -15.0, "coordinate_source": "manual_reviewed"},
}

## Celda 7 — Lectura robusta de CSV Puertos del Estado

In [21]:
def read_text_lines(path, encodings=("utf-8", "utf-8-sig", "latin1", "cp1252")):
    last_error = None

    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="replace") as f:
                return f.readlines(), enc
        except Exception as e:
            last_error = e

    raise last_error


def detect_delimiter(lines, sample_size=150):
    candidates = [";", ",", "\t", "|"]
    scores = {sep: 0 for sep in candidates}

    for line in lines[:sample_size]:
        for sep in candidates:
            scores[sep] += line.count(sep)

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else ";"


def detect_table_start(lines, delimiter):
    date_keywords = ["FECHA", "DATE", "HORA", "TIME", "GMT", "UTC"]

    for i, line in enumerate(lines[:600]):
        norm = normalize_text(line)
        if pd.isna(norm):
            continue

        has_keyword = any(k in norm for k in date_keywords)
        has_delim = line.count(delimiter) >= 1

        if has_keyword and has_delim:
            return i

    for i, line in enumerate(lines[:600]):
        if line.count(delimiter) >= 1 and looks_like_date_string(line):
            if i > 0 and lines[i - 1].count(delimiter) >= 1 and not looks_like_date_string(lines[i - 1]):
                return i - 1
            return i

    for i, line in enumerate(lines[:600]):
        if line.count(delimiter) >= 2:
            return i

    return 0


def extract_coords_and_name_from_metadata(metadata_lines):
    lat = np.nan
    lon = np.nan
    station_name = None

    for line in metadata_lines:
        norm = normalize_text(line)
        if pd.isna(norm):
            continue

        if station_name is None and any(k in norm for k in ["BOYA", "ESTACION", "ESTACIÓN", "STATION", "PUERTO", "REDEXT", "REDCOS"]):
            station_name = re.sub(r"\s+", " ", str(line)).strip()

        if "LAT" in norm and pd.isna(lat):
            candidate = parse_coordinate(line)
            if 20 <= candidate <= 40:
                lat = candidate

        if ("LON" in norm or "LONG" in norm) and pd.isna(lon):
            candidate = parse_coordinate(line)
            if -30 <= candidate <= 0 or 0 <= candidate <= 30:
                lon = candidate
                if lon > 0 and any(h in norm for h in [" W", " O", "OESTE", "WEST"]):
                    lon = -lon

    return lat, lon, station_name


def extract_coords_from_dataframe(df):
    lat_col = infer_column(df, ["lat", "latitude", "latitud"])
    lon_col = infer_column(df, ["lon", "lng", "longitude", "longitud"])

    lat = np.nan
    lon = np.nan

    if lat_col is not None:
        vals = df[lat_col].apply(parse_coordinate)
        vals = vals[vals.between(20, 40)]
        if len(vals):
            lat = float(vals.iloc[0])

    if lon_col is not None:
        vals = df[lon_col].apply(parse_coordinate)
        vals = vals[vals.between(-30, 0)]
        if len(vals):
            lon = float(vals.iloc[0])

    return lat, lon


def read_puertos_csv(path):
    lines, encoding = read_text_lines(path)
    delimiter = detect_delimiter(lines)
    table_start = detect_table_start(lines, delimiter)
    metadata_lines = lines[:table_start]

    read_kwargs = dict(
        sep=delimiter,
        skiprows=table_start,
        encoding=encoding,
        engine="python",
        on_bad_lines="skip",
    )

    df = pd.read_csv(path, **read_kwargs)

    # Si pandas tomó una fila de datos como cabecera.
    if any(looks_like_date_string(c) for c in df.columns):
        df = pd.read_csv(path, header=None, **read_kwargs)
        df.columns = [f"col_{i}" for i in range(df.shape[1])]

    df = df.dropna(axis=1, how="all")
    df.columns = [str(c).strip() for c in df.columns]

    lat_meta, lon_meta, station_name = extract_coords_and_name_from_metadata(metadata_lines)
    lat_df, lon_df = extract_coords_from_dataframe(df)

    lat = lat_meta if not pd.isna(lat_meta) else lat_df
    lon = lon_meta if not pd.isna(lon_meta) else lon_df

    meta = {
        "encoding": encoding,
        "delimiter": delimiter,
        "table_start_line": table_start,
        "metadata_line_count": len(metadata_lines),
        "lat": lat,
        "lon": lon,
        "station_name": station_name,
        "raw_columns": list(df.columns),
        "metadata_preview": "\\n".join([str(x).strip() for x in metadata_lines[:30]]),
    }

    return df, meta

## Celda 8 — Detección temporal

In [22]:
TIMESTAMP_EXCLUDE = [r"LAT", r"LON", r"LONG", r"POINT", r"PUNTO", r"ID", r"ESTACION", r"STATION"]


def is_good_timestamp(timestamp, min_valid_ratio=0.5):
    ts = pd.to_datetime(timestamp, utc=True, errors="coerce")

    if len(ts) == 0:
        return False

    if ts.notna().mean() < min_valid_ratio:
        return False

    ts_valid = ts.dropna()

    if ts_valid.empty:
        return False

    return ts_valid.between(MIN_VALID_TS, MAX_VALID_TS).mean() >= min_valid_ratio


def build_timestamp_from_filename(filename_meta, n_rows):
    start = filename_meta.get("file_start", pd.NaT)
    end = filename_meta.get("file_end", pd.NaT)

    if pd.isna(start) or pd.isna(end):
        raise ValueError(f"No se puede reconstruir timestamp desde filename: {filename_meta['filename']}")

    start = pd.to_datetime(start, utc=True).floor("min")
    end = pd.to_datetime(end, utc=True).floor("min")

    if n_rows <= 0:
        return pd.Series([], dtype="datetime64[ns, UTC]"), ["filename_fallback"], "filename_fallback"

    if n_rows == 1:
        return pd.Series([start], dtype="datetime64[ns, UTC]"), ["filename_fallback"], "filename_fallback"

    total_seconds = max((end - start).total_seconds(), n_rows - 1)
    step_seconds = total_seconds / (n_rows - 1)

    candidate_steps = np.array([60, 300, 600, 1200, 1800, 3600], dtype=float)
    nearest = candidate_steps[np.argmin(np.abs(candidate_steps - step_seconds))]

    if abs(nearest - step_seconds) / max(step_seconds, 1) < 0.25:
        freq = pd.to_timedelta(int(nearest), unit="s")
        ts = pd.date_range(start=start, periods=n_rows, freq=freq, tz="UTC")
    else:
        ts = pd.to_datetime(
            np.linspace(start.value, end.value, n_rows).astype("int64"),
            utc=True,
        )

    return pd.Series(ts, dtype="datetime64[ns, UTC]"), ["filename_inferred_frequency_fallback"], "filename_fallback"


def detect_timestamp(df, filename_meta):
    date_cols = []
    time_cols = []

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        if "FECHA" in norm or "DATE" in norm or norm in ["DIA", "DAY"]:
            date_cols.append(col)

        if "HORA" in norm or "TIME" in norm or norm in ["HH", "H"]:
            time_cols.append(col)

    for dcol in date_cols:
        compact = parse_compact_datetime_series(df[dcol])
        if compact is not None and is_good_timestamp(compact):
            return compact, [dcol], "csv_compact_date"

        if time_cols:
            for hcol in time_cols:
                candidate = df[dcol].astype(str).str.strip() + " " + df[hcol].astype(str).str.strip()

                compact = parse_compact_datetime_series(candidate)
                if compact is not None and is_good_timestamp(compact):
                    return compact, [dcol, hcol], "csv_compact_datetime"

                parsed = pd.to_datetime(candidate, errors="coerce", dayfirst=True, utc=True)
                if is_good_timestamp(parsed):
                    return parsed, [dcol, hcol], "csv_date_time_columns"

        parsed = pd.to_datetime(df[dcol], errors="coerce", dayfirst=True, utc=True)
        if is_good_timestamp(parsed):
            return parsed, [dcol], "csv_date_column"

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        compact = parse_compact_datetime_series(df[col])
        if compact is not None and is_good_timestamp(compact):
            return compact, [col], "csv_compact_any_column"

    for col in df.columns:
        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in TIMESTAMP_EXCLUDE):
            continue

        parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True, utc=True)
        if is_good_timestamp(parsed):
            return parsed, [col], "csv_any_datetime_column"

    if df.shape[1] >= 2:
        candidate = df.iloc[:, 0].astype(str).str.strip() + " " + df.iloc[:, 1].astype(str).str.strip()
        parsed = pd.to_datetime(candidate, errors="coerce", dayfirst=True, utc=True)
        if is_good_timestamp(parsed):
            return parsed, [df.columns[0], df.columns[1]], "csv_first_two_columns"

    return build_timestamp_from_filename(filename_meta, len(df))

## Celda 9 — Detección de variables de oleaje observado

In [23]:
WAVE_PATTERNS = {
    "hs": [
        r"(^|_)HS($|_)",
        r"HM0",
        r"HMO",
        r"ALT.*SIGN",
        r"SIGN.*ALT",
        r"ALTURA.*SIGNIFICATIVA",
        r"SIGNIFICANT.*HEIGHT",
    ],
    "hmax": [
        r"HMAX",
        r"H_MAX",
        r"ALT.*MAX",
        r"MAX.*OLA",
        r"MAXIMUM.*HEIGHT",
    ],
    "tp": [
        r"(^|_)TP($|_)",
        r"PER.*PICO",
        r"PEAK.*PER",
        r"VTPK",
        r"TPK",
    ],
    "tm02": [
        r"TM02",
        r"T_M02",
        r"PER.*MED",
        r"PERIODO.*MEDIO",
        r"MEAN.*PER",
        r"TZ",
        r"TM",
    ],
    "wave_direction": [
        r"DIR.*MED",
        r"DIREC.*MED",
        r"DIRECCION.*OLEAJE",
        r"DIRECCION.*OLA",
        r"DIR.*OLA",
        r"PROCED",
        r"MEAN.*DIR",
        r"VMDR",
        r"(^|_)DIR($|_)",
    ],
    "swell_height": [
        r"SWELL.*H",
        r"MAR.*FONDO.*ALT",
    ],
    "swell_period": [
        r"SWELL.*PER",
        r"MAR.*FONDO.*PER",
    ],
    "swell_direction": [
        r"SWELL.*DIR",
        r"MAR.*FONDO.*DIR",
        r"MAR.*FONDO.*DIREC",
    ],
    "wind_wave_height": [
        r"WIND.*WAVE.*H",
        r"SEA.*H",
        r"MAR.*VIENTO.*ALT",
    ],
    "wind_wave_period": [
        r"WIND.*WAVE.*PER",
        r"SEA.*PER",
        r"MAR.*VIENTO.*PER",
    ],
}


def map_wave_columns(df, timestamp_cols):
    excludes = [
        r"^FECHA",
        r"^DATE",
        r"^HORA",
        r"^TIME",
        r"^LAT",
        r"^LON",
        r"^LONG",
        r"^ID$",
        r"^ESTACION",
        r"^STATION",
        r"^BOYA",
    ]

    mapped = {}
    used_cols = set(timestamp_cols)

    for canonical, patterns in WAVE_PATTERNS.items():
        col = find_col_by_patterns(
            df,
            patterns,
            exclude_patterns=excludes,
            used_cols=used_cols,
        )

        if col is not None:
            mapped[canonical] = col
            used_cols.add(col)

    numeric_candidates = []

    for col in df.columns:
        if col in used_cols:
            continue

        norm = normalize_col(col)

        if any(re.search(ex, norm) for ex in excludes):
            continue

        vals = to_numeric_series(df[col])
        if vals.notna().mean() > 0.5:
            numeric_candidates.append(col)

    # Fallback conservador: si no detecta hs, usar primera columna numérica.
    if "hs" not in mapped and len(numeric_candidates) >= 1:
        mapped["hs"] = numeric_candidates[0]
        used_cols.add(numeric_candidates[0])

    return mapped, numeric_candidates


def normalize_wave_units(series, raw_col_name, canonical):
    values = to_numeric_series(series)
    col_norm = normalize_col(raw_col_name)

    # Alturas: m por defecto; convertir cm/mm si el nombre lo indica o magnitud muy grande.
    if canonical in ["hs", "hmax", "swell_height", "wind_wave_height"]:
        if "MM" in col_norm:
            values = values / 1000.0
            unit = "millimeters_to_meters_from_column_name"
        elif "CM" in col_norm:
            values = values / 100.0
            unit = "centimeters_to_meters_from_column_name"
        else:
            q99 = values.abs().quantile(0.99)
            if pd.notna(q99) and q99 > 1000:
                values = values / 1000.0
                unit = "millimeters_to_meters_by_magnitude"
            elif pd.notna(q99) and q99 > 50:
                values = values / 100.0
                unit = "centimeters_to_meters_by_magnitude"
            else:
                unit = "meters"

    # Periodos en segundos por defecto.
    elif canonical in ["tp", "tm02", "swell_period", "wind_wave_period"]:
        unit = "seconds"

    # Direcciones en grados.
    elif canonical in ["wave_direction", "swell_direction"]:
        values = values % 360
        unit = "degrees"

    else:
        unit = "unknown"

    return values, unit

celda 9b

In [24]:
def make_unique_columns(columns):
    """
    Evita columnas duplicadas.
    Si aparece dos veces 'Periodo Medio(s)', genera:
    'Periodo Medio(s)' y 'Periodo Medio(s)__dup1'
    """
    seen = {}
    unique = []

    for col in columns:
        col = str(col).strip()

        if col not in seen:
            seen[col] = 0
            unique.append(col)
        else:
            seen[col] += 1
            unique.append(f"{col}__dup{seen[col]}")

    return unique


def to_numeric_series(series):
    """
    Versión robusta:
    - Si recibe DataFrame por columnas duplicadas, usa la primera columna.
    - Convierte coma decimal.
    - Limpia códigos missing.
    """
    if series is None:
        return pd.Series(dtype="float64")

    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]

    s = series.astype(str).str.strip()

    missing_tokens = {
        "",
        "NA",
        "N/A",
        "NAN",
        "NULL",
        "NONE",
        "-",
        "--",
        "---",
        "S/D",
        "SD",
        "NODATA",
        "NO_DATA",
    }

    s = s.mask(s.str.upper().isin(missing_tokens))
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9eE+\-.]", "", regex=True)

    out = pd.to_numeric(s, errors="coerce")

    out = out.mask(out.isin([-99999, -9999, -999, 999, 9999, 99999]))
    out = out.mask(out <= -999)

    return out


def read_puertos_csv(path):
    """
    Versión parcheada:
    - Lee CSV Puertos del Estado.
    - Detecta cabecera.
    - Hace únicos los nombres de columnas para evitar errores con duplicados.
    """
    lines, encoding = read_text_lines(path)
    delimiter = detect_delimiter(lines)
    table_start = detect_table_start(lines, delimiter)
    metadata_lines = lines[:table_start]

    read_kwargs = dict(
        sep=delimiter,
        skiprows=table_start,
        encoding=encoding,
        engine="python",
        on_bad_lines="skip",
    )

    df = pd.read_csv(path, **read_kwargs)

    # Si pandas tomó una fila de datos como cabecera.
    if any(looks_like_date_string(c) for c in df.columns):
        df = pd.read_csv(path, header=None, **read_kwargs)
        df.columns = [f"col_{i}" for i in range(df.shape[1])]

    df = df.dropna(axis=1, how="all")
    df.columns = make_unique_columns([str(c).strip() for c in df.columns])

    lat_meta, lon_meta, station_name = extract_coords_and_name_from_metadata(metadata_lines)
    lat_df, lon_df = extract_coords_from_dataframe(df)

    lat = lat_meta if not pd.isna(lat_meta) else lat_df
    lon = lon_meta if not pd.isna(lon_meta) else lon_df

    meta = {
        "encoding": encoding,
        "delimiter": delimiter,
        "table_start_line": table_start,
        "metadata_line_count": len(metadata_lines),
        "lat": lat,
        "lon": lon,
        "station_name": station_name,
        "raw_columns": list(df.columns),
        "metadata_preview": "\n".join([str(x).strip() for x in metadata_lines[:30]]),
    }

    return df, meta


print("Parche 9B cargado: columnas duplicadas y to_numeric_series robustos.")

Parche 9B cargado: columnas duplicadas y to_numeric_series robustos.


## Celda 10 — Estandarizar y resamplear archivos a horario

In [25]:
def circular_mean_degrees(x):
    x = pd.to_numeric(x, errors="coerce").dropna()

    if len(x) == 0:
        return np.nan

    radians = np.deg2rad(x)
    sin_mean = np.sin(radians).mean()
    cos_mean = np.cos(radians).mean()

    if np.isclose(sin_mean, 0) and np.isclose(cos_mean, 0):
        return np.nan

    return (np.degrees(np.arctan2(sin_mean, cos_mean)) + 360) % 360


def resample_validation_hourly(df):
    df = df.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="first").copy()
    df = df.set_index("timestamp")

    cols = [c for c in [
        "hs",
        "hmax",
        "tp",
        "tm02",
        "wave_direction",
        "swell_height",
        "swell_period",
        "swell_direction",
        "wind_wave_height",
        "wind_wave_period",
    ] if c in df.columns]

    agg_dict = {}

    for col in cols:
        if col in ["wave_direction", "swell_direction"]:
            agg_dict[col] = circular_mean_degrees
        elif col == "hmax":
            agg_dict[col] = "max"
        else:
            agg_dict[col] = "mean"

    hourly = df[cols].resample("1h").agg(agg_dict)
    hourly["raw_observations_in_hour"] = df["hs"].resample("1h").count() if "hs" in df.columns else 0

    hourly = hourly.reset_index()

    return hourly


def standardize_redext_file(path, filename_meta):
    raw_df, file_meta = read_puertos_csv(path)

    timestamp, timestamp_cols, timestamp_source = detect_timestamp(raw_df, filename_meta)

    mapped_cols, numeric_candidates = map_wave_columns(raw_df, timestamp_cols)

    if not mapped_cols:
        raise ValueError(f"{path.name}: no se detectó ninguna variable de oleaje.")

    out = pd.DataFrame()
    out["timestamp"] = pd.to_datetime(timestamp, utc=True, errors="coerce").reset_index(drop=True)
    out["station_id"] = str(filename_meta["station_id"])
    out["source_file"] = filename_meta["filename"]

    unit_map = {}

    for canonical, raw_col in mapped_cols.items():
        values, unit = normalize_wave_units(raw_df[raw_col], raw_col, canonical)
        out[canonical] = values.reset_index(drop=True)
        unit_map[canonical] = {
            "raw_col": raw_col,
            "unit_inferred": unit,
        }

    out = out.dropna(subset=["timestamp"]).copy()

    invalid_time = ~out["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

    if invalid_time.any():
        raise ValueError(
            f"{path.name}: timestamps fuera de rango "
            f"{out.loc[invalid_time, 'timestamp'].min()} - "
            f"{out.loc[invalid_time, 'timestamp'].max()}"
        )

    hourly = resample_validation_hourly(out)

    station_id = str(filename_meta["station_id"])

    lat = file_meta.get("lat", np.nan)
    lon = file_meta.get("lon", np.nan)
    station_name = file_meta.get("station_name", None)
    coordinate_source = "csv_metadata_or_columns"

    fallback = REDEXT_REDCOS_COORD_FALLBACK.get(station_id)

    if (pd.isna(lat) or pd.isna(lon)) and fallback is not None:
        lat = fallback.get("lat", lat)
        lon = fallback.get("lon", lon)
        station_name = station_name or fallback.get("station_name")
        coordinate_source = fallback.get("coordinate_source", "manual_fallback")

    hourly["station_id"] = station_id
    hourly["station_name"] = station_name if station_name is not None else f"REDEXT_REDCOS_{station_id}"
    hourly["lat"] = lat
    hourly["lon"] = lon
    hourly["source_file"] = filename_meta["filename"]
    hourly["coordinate_source"] = coordinate_source if not (pd.isna(lat) or pd.isna(lon)) else "missing_coordinates"

    summary = {
        "filename": filename_meta["filename"],
        "station_id": station_id,
        "variable_group": filename_meta["variable_group"],
        "raw_rows": len(raw_df),
        "hourly_rows": len(hourly),
        "timestamp_min": hourly["timestamp"].min() if len(hourly) else pd.NaT,
        "timestamp_max": hourly["timestamp"].max() if len(hourly) else pd.NaT,
        "lat": lat,
        "lon": lon,
        "coordinate_source": hourly["coordinate_source"].iloc[0] if len(hourly) else coordinate_source,
        "station_name": station_name,
        "timestamp_source": timestamp_source,
        "mapped_columns": json.dumps(mapped_cols, ensure_ascii=False),
        "unit_map": json.dumps(unit_map, ensure_ascii=False),
        "numeric_candidate_columns": json.dumps(numeric_candidates, ensure_ascii=False),
        "raw_columns": json.dumps(file_meta.get("raw_columns", []), ensure_ascii=False),
        "metadata_preview": file_meta.get("metadata_preview", ""),
        "encoding": file_meta.get("encoding"),
        "delimiter": file_meta.get("delimiter"),
        "table_start_line": file_meta.get("table_start_line"),
    }

    return hourly, summary


processed_frames = []
file_summaries = []
read_errors = []

for _, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Procesando REDEXT/REDCOS"):
    path = Path(row["path"])
    filename_meta = row.drop(labels=["path"]).to_dict()

    try:
        hourly, summary = standardize_redext_file(path, filename_meta)
        processed_frames.append(hourly)
        file_summaries.append(summary)

    except Exception as e:
        read_errors.append(
            {
                "filename": path.name,
                "path": str(path),
                "error": repr(e),
            }
        )

file_summary_df = pd.DataFrame(file_summaries)
read_errors_df = pd.DataFrame(read_errors)

print("Archivos procesados correctamente:", len(file_summary_df))
print("Errores de lectura:", len(read_errors_df))

display(file_summary_df)
display(read_errors_df)

file_summary_df.to_csv(QC_DIR / "quality_redext_redcos_file_summary.csv", index=False)
read_errors_df.to_csv(QC_DIR / "quality_redext_redcos_read_errors.csv", index=False)

if len(read_errors_df):
    raise ValueError("Hay errores leyendo REDEXT/REDCOS. Revisar quality_redext_redcos_read_errors.csv.")

if not processed_frames:
    raise ValueError("No se procesó ningún archivo REDEXT/REDCOS.")

validation_raw = pd.concat(processed_frames, ignore_index=True)

print("validation_raw shape:", validation_raw.shape)
display(validation_raw.head())

Procesando REDEXT/REDCOS:   0%|          | 0/5 [00:00<?, ?it/s]

Archivos procesados correctamente: 5
Errores de lectura: 0


,filename,station_id,variable_group,raw_rows,hourly_rows,timestamp_min,timestamp_max,lat,lon,coordinate_source,station_name,timestamp_source,mapped_columns,unit_map,numeric_candidate_columns,raw_columns,metadata_preview,encoding,delimiter,table_start_line
0,25411_52356_2442_ALL_20010101184619_2026050617...,2442,ALL,190386,222168,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,NaN,NaN,missing_coordinates,None,csv_compact_date,"{""hs"": ""Altura Signif. del Oleaje(m)"", ""hmax"":...","{""hs"": {""raw_col"": ""Altura Signif. del Oleaje(...","[""Periodo de la Ola Maxima(s)"", ""Canal de obte...","[""Fecha (GMT)"", ""Altura Signif. del Oleaje(m)""...",Valor nulo: -9999.9,utf-8,\t,1
1,25411_52357_1440_ALL_20070620220000_2014012423...,1440,ALL,53852,57853,2007-06-20 11:00:00+00:00,2014-01-24 23:00:00+00:00,NaN,NaN,missing_coordinates,None,csv_compact_date,"{""hs"": ""Altura Signif. del Oleaje(m)"", ""hmax"":...","{""hs"": {""raw_col"": ""Altura Signif. del Oleaje(...","[""Altura signif. de cruce por cero(m)"", ""Perio...","[""Fecha (GMT)"", ""Altura Signif. del Oleaje(m)""...",Valor nulo: -9999.9,utf-8,\t,1
2,25411_52358_2446_ALL_20010101184650_2026050617...,2446,ALL,202123,219472,2001-04-23 08:00:00+00:00,2026-05-06 23:00:00+00:00,NaN,NaN,missing_coordinates,None,csv_compact_date,"{""hs"": ""Altura Signif. del Oleaje(m)"", ""hmax"":...","{""hs"": {""raw_col"": ""Altura Signif. del Oleaje(...","[""Periodo de la Ola Maxima(s)"", ""Canal de obte...","[""Fecha (GMT)"", ""Altura Signif. del Oleaje(m)""...",Valor nulo: -9999.9,utf-8,\t,1
3,25411_52359_1414_ALL_20010101184657_2026050617...,1414,ALL,207024,222168,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,NaN,NaN,missing_coordinates,None,csv_compact_date,"{""hs"": ""Altura Signif. del Oleaje (Hm0)(m)"", ""...","{""hs"": {""raw_col"": ""Altura Signif. del Oleaje ...","[""Altura signif. de cruce por cero (H1/3)(m)"",...","[""Fecha (GMT)"", ""Altura Signif. del Oleaje (Hm...",Valor nulo: -9999.9,utf-8,\t,1
4,25411_52360_1421_ALL_20250506174709_2026050617...,1421,ALL,7546,8784,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,NaN,NaN,missing_coordinates,None,csv_compact_date,"{""hs"": ""Altura Signif. del Oleaje (Hm0)(m)"", ""...","{""hs"": {""raw_col"": ""Altura Signif. del Oleaje ...","[""Altura signif. de cruce por cero (H1/3)(m)"",...","[""Fecha (GMT)"", ""Altura Signif. del Oleaje (Hm...",Valor nulo: -9999.9,utf-8,\t,1


""


validation_raw shape: (730445, 13)


,timestamp,hs,hmax,tp,tm02,wave_direction,raw_observations_in_hour,station_id,station_name,lat,lon,source_file,coordinate_source
0,2001-01-01 00:00:00+00:00,1.22,1.68,12.13,6.69,NaN,1,2442,REDEXT_REDCOS_2442,NaN,NaN,25411_52356_2442_ALL_20010101184619_2026050617...,missing_coordinates
1,2001-01-01 01:00:00+00:00,1.16,1.75,11.16,6.40,NaN,1,2442,REDEXT_REDCOS_2442,NaN,NaN,25411_52356_2442_ALL_20010101184619_2026050617...,missing_coordinates
2,2001-01-01 02:00:00+00:00,1.24,1.96,11.63,6.42,NaN,1,2442,REDEXT_REDCOS_2442,NaN,NaN,25411_52356_2442_ALL_20010101184619_2026050617...,missing_coordinates
3,2001-01-01 03:00:00+00:00,1.16,1.82,11.63,6.16,NaN,1,2442,REDEXT_REDCOS_2442,NaN,NaN,25411_52356_2442_ALL_20010101184619_2026050617...,missing_coordinates
4,2001-01-01 04:00:00+00:00,1.21,1.82,10.66,6.29,NaN,1,2442,REDEXT_REDCOS_2442,NaN,NaN,25411_52356_2442_ALL_20010101184619_2026050617...,missing_coordinates


## Celda 11 — Metadata de estaciones y asignación a zonas

In [26]:
station_meta = (
    validation_raw[
        [
            "station_id",
            "station_name",
            "lat",
            "lon",
            "coordinate_source",
        ]
    ]
    .drop_duplicates(subset=["station_id"])
    .copy()
)

station_meta["coords_missing"] = station_meta["lat"].isna() | station_meta["lon"].isna()

station_meta["inside_bbox"] = (
    station_meta["lat"].between(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"])
    & station_meta["lon"].between(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"])
)

print("Estaciones validación:", len(station_meta))
print("Estaciones sin coordenadas:", int(station_meta["coords_missing"].sum()))
print("Estaciones dentro bbox:", int(station_meta["inside_bbox"].sum()))

display(station_meta)

# Asignar zonas solo a estaciones con coordenadas.
station_zone_parts = []

with_coords = station_meta[~station_meta["coords_missing"]].copy()

if len(with_coords):
    gdf_stations = gpd.GeoDataFrame(
        with_coords.copy(),
        geometry=gpd.points_from_xy(with_coords["lon"], with_coords["lat"]),
        crs="EPSG:4326",
    )

    gdf_stations_m = gdf_stations.to_crs("EPSG:3857")

    nearest = gpd.sjoin_nearest(
        gdf_stations_m,
        gdf_zones_m[["zona_id", "nombre_zona", "isla", "municipio", "geometry"]],
        how="left",
        distance_col="distance_to_zona_m",
    )

    nearest = (
        nearest
        .sort_values("distance_to_zona_m")
        .groupby("station_id", as_index=False)
        .first()
    )

    station_zone_with_coords = pd.DataFrame(nearest.drop(columns="geometry", errors="ignore"))
    station_zone_with_coords["distance_to_zona_km"] = station_zone_with_coords["distance_to_zona_m"] / 1000
    station_zone_parts.append(station_zone_with_coords)

without_coords = station_meta[station_meta["coords_missing"]].copy()

if len(without_coords):
    station_zone_without_coords = without_coords.copy()
    station_zone_without_coords["zona_id"] = "CAN_VALIDATION_UNASSIGNED"
    station_zone_without_coords["nombre_zona"] = "VALIDATION_UNASSIGNED"
    station_zone_without_coords["isla"] = "ISLA_DESCONOCIDA"
    station_zone_without_coords["municipio"] = "MUNICIPIO_DESCONOCIDO"
    station_zone_without_coords["distance_to_zona_km"] = np.nan
    station_zone_parts.append(station_zone_without_coords)

station_zone = pd.concat(station_zone_parts, ignore_index=True)

needed_cols = [
    "station_id",
    "station_name",
    "lat",
    "lon",
    "coordinate_source",
    "zona_id",
    "nombre_zona",
    "isla",
    "municipio",
    "distance_to_zona_km",
]

for c in needed_cols:
    if c not in station_zone.columns:
        station_zone[c] = np.nan

station_zone = station_zone[needed_cols].copy()

display(station_zone)

station_zone.to_csv(META_DIR / "redext_redcos_station_to_zone.csv", index=False)

if station_zone["distance_to_zona_km"].notna().any():
    print("Distancia estación → zona más cercana, km:")
    display(station_zone["distance_to_zona_km"].describe())

if station_meta["coords_missing"].any():
    print(
        "AVISO: algunas boyas no tienen coordenadas. "
        "Se guardan como CAN_VALIDATION_UNASSIGNED y quedan documentadas."
    )

Estaciones validación: 5
Estaciones sin coordenadas: 5
Estaciones dentro bbox: 0


,station_id,station_name,lat,lon,coordinate_source,coords_missing,inside_bbox
0,2442,REDEXT_REDCOS_2442,NaN,NaN,missing_coordinates,True,False
222168,1440,REDEXT_REDCOS_1440,NaN,NaN,missing_coordinates,True,False
280021,2446,REDEXT_REDCOS_2446,NaN,NaN,missing_coordinates,True,False
499493,1414,REDEXT_REDCOS_1414,NaN,NaN,missing_coordinates,True,False
721661,1421,REDEXT_REDCOS_1421,NaN,NaN,missing_coordinates,True,False


,station_id,station_name,lat,lon,coordinate_source,zona_id,nombre_zona,isla,municipio,distance_to_zona_km
0,2442,REDEXT_REDCOS_2442,NaN,NaN,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN
1,1440,REDEXT_REDCOS_1440,NaN,NaN,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN
2,2446,REDEXT_REDCOS_2446,NaN,NaN,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN
3,1414,REDEXT_REDCOS_1414,NaN,NaN,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN
4,1421,REDEXT_REDCOS_1421,NaN,NaN,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN


AVISO: algunas boyas no tienen coordenadas. Se guardan como CAN_VALIDATION_UNASSIGNED y quedan documentadas.


## Celda 12 — Construir tabla final de validación

In [27]:
validation_silver = validation_raw.merge(
    station_zone[
        [
            "station_id",
            "zona_id",
            "nombre_zona",
            "isla",
            "municipio",
            "distance_to_zona_km",
        ]
    ],
    on="station_id",
    how="left",
)

validation_silver["source"] = SOURCE_NAME
validation_silver["validation_role"] = "observed_wave_buoy_validation"
validation_silver["temporal_resolution"] = "hourly"
validation_silver["year"] = validation_silver["timestamp"].dt.year.astype("Int64")

# Si faltan zonas por cualquier razón, no romper el pipeline de validación.
validation_silver["zona_id"] = validation_silver["zona_id"].fillna("CAN_VALIDATION_UNASSIGNED")
validation_silver["nombre_zona"] = validation_silver["nombre_zona"].fillna("VALIDATION_UNASSIGNED")
validation_silver["isla"] = validation_silver["isla"].fillna("ISLA_DESCONOCIDA")
validation_silver["municipio"] = validation_silver["municipio"].fillna("MUNICIPIO_DESCONOCIDO")

before = len(validation_silver)

validation_silver = (
    validation_silver
    .sort_values(["station_id", "timestamp", "source_file"])
    .drop_duplicates(subset=["station_id", "timestamp", "source"], keep="first")
    .copy()
)

duplicates_removed = before - len(validation_silver)

print("validation_silver shape:", validation_silver.shape)
print("Duplicados eliminados:", duplicates_removed)
display(validation_silver.head())

validation_silver shape: (730445, 22)
Duplicados eliminados: 0


,timestamp,hs,hmax,tp,tm02,wave_direction,raw_observations_in_hour,station_id,station_name,lat,...,coordinate_source,zona_id,nombre_zona,isla,municipio,distance_to_zona_km,source,validation_role,temporal_resolution,year
499493,2001-01-01 00:00:00+00:00,0.81,1.29,11.14,5.16,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001
499494,2001-01-01 01:00:00+00:00,0.85,1.29,11.63,5.45,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001
499495,2001-01-01 02:00:00+00:00,0.82,1.24,11.61,5.34,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001
499496,2001-01-01 03:00:00+00:00,0.79,1.16,10.66,5.22,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001
499497,2001-01-01 04:00:00+00:00,0.81,1.42,11.63,5.41,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,missing_coordinates,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,ISLA_DESCONOCIDA,MUNICIPIO_DESCONOCIDO,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001


## Celda 13 — Flags de calidad

In [28]:
VARIABLE_RANGES = {
    "hs": (0, 15),
    "hmax": (0, 25),
    "tp": (0, 35),
    "tm02": (0, 35),
    "wave_direction": (0, 360),
    "swell_height": (0, 15),
    "swell_period": (0, 35),
    "swell_direction": (0, 360),
    "wind_wave_height": (0, 15),
    "wind_wave_period": (0, 35),
}


def add_quality_flags(df, variable_ranges):
    df = df.copy()

    for col, (vmin, vmax) in variable_ranges.items():
        if col not in df.columns:
            continue

        flag_col = f"{col}_flag"
        df[flag_col] = 0

        missing_mask = df[col].isna()
        outlier_mask = (~missing_mask) & ((df[col] < vmin) | (df[col] > vmax))

        df.loc[missing_mask, flag_col] = 1
        df.loc[outlier_mask, flag_col] = 2

        df[flag_col] = df[flag_col].astype("int8")

    return df


validation_silver = add_quality_flags(validation_silver, VARIABLE_RANGES)

display(validation_silver.head())

,timestamp,hs,hmax,tp,tm02,wave_direction,raw_observations_in_hour,station_id,station_name,lat,...,distance_to_zona_km,source,validation_role,temporal_resolution,year,hs_flag,hmax_flag,tp_flag,tm02_flag,wave_direction_flag
499493,2001-01-01 00:00:00+00:00,0.81,1.29,11.14,5.16,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001,0,0,0,0,1
499494,2001-01-01 01:00:00+00:00,0.85,1.29,11.63,5.45,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001,0,0,0,0,1
499495,2001-01-01 02:00:00+00:00,0.82,1.24,11.61,5.34,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001,0,0,0,0,1
499496,2001-01-01 03:00:00+00:00,0.79,1.16,10.66,5.22,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001,0,0,0,0,1
499497,2001-01-01 04:00:00+00:00,0.81,1.42,11.63,5.41,NaN,1,1414,REDEXT_REDCOS_1414,NaN,...,NaN,REDEXT_REDCOS,observed_wave_buoy_validation,hourly,2001,0,0,0,0,1


## Celda 14 — Reportes de calidad y gaps

In [29]:
def gap_report_hourly(df, station_col="station_id", timestamp_col="timestamp", expected_hours=1):
    rows = []

    for station_id, g in df[[station_col, timestamp_col]].dropna().groupby(station_col):
        ts = g[timestamp_col].sort_values().drop_duplicates()
        diffs_h = ts.diff().dropna().dt.total_seconds() / 3600
        gaps = diffs_h[diffs_h > expected_hours * 1.5]

        rows.append(
            {
                "station_id": station_id,
                "timestamp_min": ts.min(),
                "timestamp_max": ts.max(),
                "rows": len(ts),
                "gaps_count": int(len(gaps)),
                "max_gap_hours": float(gaps.max()) if len(gaps) else 0.0,
                "expected_hours": expected_hours,
            }
        )

    return pd.DataFrame(rows)


quality_summary = pd.DataFrame(
    [
        {
            "table": "ocean_validation",
            "source": SOURCE_NAME,
            "rows": len(validation_silver),
            "unique_stations": validation_silver["station_id"].nunique(),
            "unique_zona_id": validation_silver["zona_id"].nunique(),
            "timestamp_min": validation_silver["timestamp"].min(),
            "timestamp_max": validation_silver["timestamp"].max(),
            "duplicates_removed": duplicates_removed,
            "coords_missing_station_count": int(station_meta["coords_missing"].sum()),
            "hs_missing_pct": float(validation_silver["hs"].isna().mean() * 100) if "hs" in validation_silver.columns else 100.0,
            "hmax_missing_pct": float(validation_silver["hmax"].isna().mean() * 100) if "hmax" in validation_silver.columns else 100.0,
            "tp_missing_pct": float(validation_silver["tp"].isna().mean() * 100) if "tp" in validation_silver.columns else 100.0,
            "tm02_missing_pct": float(validation_silver["tm02"].isna().mean() * 100) if "tm02" in validation_silver.columns else 100.0,
            "wave_direction_missing_pct": float(validation_silver["wave_direction"].isna().mean() * 100) if "wave_direction" in validation_silver.columns else 100.0,
        }
    ]
)

missing_by_column = (
    validation_silver.isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

gaps_by_station = gap_report_hourly(validation_silver)

display(quality_summary)
display(missing_by_column)
display(gaps_by_station)

quality_summary.to_csv(QC_DIR / "quality_redext_redcos_validation_summary.csv", index=False)
missing_by_column.to_csv(QC_DIR / "quality_redext_redcos_missing_by_column.csv", index=False)
gaps_by_station.to_csv(QC_DIR / "quality_redext_redcos_gaps_by_station.csv", index=False)

,table,source,rows,unique_stations,unique_zona_id,timestamp_min,timestamp_max,duplicates_removed,coords_missing_station_count,hs_missing_pct,hmax_missing_pct,tp_missing_pct,tm02_missing_pct,wave_direction_missing_pct
0,ocean_validation,REDEXT_REDCOS,730445,5,1,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,0,5,10.466633,10.341504,10.106442,10.106579,37.167891


,column,missing_pct
0,timestamp,0.000000
1,hs,10.466633
2,hmax,10.341504
3,tp,10.106442
4,tm02,10.106579
5,wave_direction,37.167891
6,raw_observations_in_hour,0.000000
7,station_id,0.000000
8,station_name,0.000000
9,lat,100.000000


,station_id,timestamp_min,timestamp_max,rows,gaps_count,max_gap_hours,expected_hours
0,1414,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,222168,0,0.0,1
1,1421,2025-05-06 00:00:00+00:00,2026-05-06 23:00:00+00:00,8784,0,0.0,1
2,1440,2007-06-20 11:00:00+00:00,2014-01-24 23:00:00+00:00,57853,0,0.0,1
3,2442,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,222168,0,0.0,1
4,2446,2001-04-23 08:00:00+00:00,2026-05-06 23:00:00+00:00,219472,0,0.0,1


## Celda 15 — Validaciones finales

In [30]:
required_cols = [
    "timestamp",
    "station_id",
    "station_name",
    "zona_id",
    "lat",
    "lon",
    "source",
    "hs",
    "hmax",
    "tp",
    "tm02",
    "wave_direction",
    "raw_observations_in_hour",
    "validation_role",
    "temporal_resolution",
    "year",
    "isla",
]

for col in required_cols:
    if col not in validation_silver.columns:
        validation_silver[col] = np.nan

if validation_silver.empty:
    raise ValueError("validation_silver está vacío.")

if validation_silver["timestamp"].isna().any():
    raise ValueError("Hay timestamps nulos.")

invalid_time = ~validation_silver["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

if invalid_time.any():
    raise ValueError(
        "Hay timestamps fuera de rango: "
        f"{validation_silver.loc[invalid_time, 'timestamp'].min()} - "
        f"{validation_silver.loc[invalid_time, 'timestamp'].max()}"
    )

if validation_silver["station_id"].isna().any():
    raise ValueError("Hay station_id nulos.")

if "hs" not in validation_silver.columns or validation_silver["hs"].isna().mean() > 0.5:
    print(
        "AVISO: hs tiene más del 50% de nulos o no se detectó correctamente. "
        "Revisar quality_redext_redcos_file_summary.csv y mapped_columns."
    )

if validation_silver["zona_id"].isna().any():
    raise ValueError("Hay zona_id nulos incluso tras fallback.")

print("Validaciones finales REDEXT/REDCOS superadas.")

Validaciones finales REDEXT/REDCOS superadas.


## Celda 16 — Guardar Parquet particionado

In [31]:
final_cols = [
    "timestamp",
    "station_id",
    "station_name",
    "zona_id",
    "nombre_zona",
    "lat",
    "lon",
    "source",
    "hs",
    "hmax",
    "tp",
    "tm02",
    "wave_direction",
    "swell_height",
    "swell_period",
    "swell_direction",
    "wind_wave_height",
    "wind_wave_period",
    "raw_observations_in_hour",
    "distance_to_zona_km",
    "validation_role",
    "temporal_resolution",
    "year",
    "isla",
    "municipio",
    "coordinate_source",
    "source_file",
]

flag_cols = [c for c in validation_silver.columns if c.endswith("_flag")]
final_cols = final_cols + flag_cols

for col in final_cols:
    if col not in validation_silver.columns:
        validation_silver[col] = np.nan

validation_final = validation_silver[final_cols].copy()

remove_existing_source_partition(OUT_VALIDATION_DIR, SOURCE_NAME)
write_partitioned_parquet(validation_final, OUT_VALIDATION_DIR)

print("Guardado REDEXT/REDCOS validation en:")
print(OUT_VALIDATION_DIR / f"source={SOURCE_NAME}")

Guardado REDEXT/REDCOS validation en:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_validation/source=REDEXT_REDCOS


## Celda 17 — Comprobación final de lectura

In [32]:
validation_count, validation_sample = dataset_count_and_sample(OUT_VALIDATION_DIR)

print("Filas guardadas ocean_validation REDEXT/REDCOS:", validation_count)

if len(validation_sample):
    display(validation_sample)

if validation_count == 0:
    raise ValueError("No se guardó ningún registro REDEXT/REDCOS.")

global_summary = pd.DataFrame(
    [
        {
            "table": "ocean_validation",
            "source": SOURCE_NAME,
            "rows": validation_count,
            "stations": validation_final["station_id"].nunique(),
            "timestamp_min": validation_final["timestamp"].min(),
            "timestamp_max": validation_final["timestamp"].max(),
            "hs_missing_pct": float(validation_final["hs"].isna().mean() * 100),
            "tp_missing_pct": float(validation_final["tp"].isna().mean() * 100),
            "wave_direction_missing_pct": float(validation_final["wave_direction"].isna().mean() * 100),
            "coords_missing_station_count": int(station_meta["coords_missing"].sum()),
        }
    ]
)

display(global_summary)

global_summary.to_csv(QC_DIR / "quality_redext_redcos_global_summary.csv", index=False)

print("Reportes REDEXT/REDCOS:")
for p in sorted(QC_DIR.glob("quality_redext_redcos*.csv")):
    print("-", p)

print("\nMetadatos REDEXT/REDCOS:")
for p in sorted(META_DIR.glob("redext_redcos*.csv")):
    print("-", p)

print("\nValidación final REDEXT/REDCOS superada.")

Filas guardadas ocean_validation REDEXT/REDCOS: 730445


,timestamp,station_id,station_name,zona_id,nombre_zona,lat,lon,hs,hmax,tp,...,coordinate_source,source_file,hs_flag,hmax_flag,tp_flag,tm02_flag,wave_direction_flag,source,year,isla
0,2001-01-01 00:00:00+00:00,1414,REDEXT_REDCOS_1414,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,NaN,NaN,0.81,1.29,11.14,...,missing_coordinates,25411_52359_1414_ALL_20010101184657_2026050617...,0,0,0,0,1,REDEXT_REDCOS,2001,ISLA_DESCONOCIDA
1,2001-01-01 01:00:00+00:00,1414,REDEXT_REDCOS_1414,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,NaN,NaN,0.85,1.29,11.63,...,missing_coordinates,25411_52359_1414_ALL_20010101184657_2026050617...,0,0,0,0,1,REDEXT_REDCOS,2001,ISLA_DESCONOCIDA
2,2001-01-01 02:00:00+00:00,1414,REDEXT_REDCOS_1414,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,NaN,NaN,0.82,1.24,11.61,...,missing_coordinates,25411_52359_1414_ALL_20010101184657_2026050617...,0,0,0,0,1,REDEXT_REDCOS,2001,ISLA_DESCONOCIDA
3,2001-01-01 03:00:00+00:00,1414,REDEXT_REDCOS_1414,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,NaN,NaN,0.79,1.16,10.66,...,missing_coordinates,25411_52359_1414_ALL_20010101184657_2026050617...,0,0,0,0,1,REDEXT_REDCOS,2001,ISLA_DESCONOCIDA
4,2001-01-01 04:00:00+00:00,1414,REDEXT_REDCOS_1414,CAN_VALIDATION_UNASSIGNED,VALIDATION_UNASSIGNED,NaN,NaN,0.81,1.42,11.63,...,missing_coordinates,25411_52359_1414_ALL_20010101184657_2026050617...,0,0,0,0,1,REDEXT_REDCOS,2001,ISLA_DESCONOCIDA


,table,source,rows,stations,timestamp_min,timestamp_max,hs_missing_pct,tp_missing_pct,wave_direction_missing_pct,coords_missing_station_count
0,ocean_validation,REDEXT_REDCOS,730445,5,2001-01-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,10.466633,10.106442,37.167891,5


Reportes REDEXT/REDCOS:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redext_redcos_file_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redext_redcos_gaps_by_station.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redext_redcos_global_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redext_redcos_missing_by_column.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redext_redcos_read_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_redext_redcos_validation_summary.csv

Metadatos REDEXT/REDCOS:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_metadata/redext_redcos_station_to_zone.csv

Validación final REDEXT/REDCOS superada.


## Resultado esperado

Al terminar deberían existir:

```text
silver/ocean_validation/source=REDEXT_REDCOS/year=YYYY/isla=.../*.parquet
silver/_quality_reports/quality_redext_redcos_global_summary.csv
silver/_quality_reports/quality_redext_redcos_validation_summary.csv
silver/_quality_reports/quality_redext_redcos_missing_by_column.csv
silver/_quality_reports/quality_redext_redcos_gaps_by_station.csv
silver/_metadata/redext_redcos_station_to_zone.csv
```

Comprueba especialmente:

```text
Archivos procesados correctamente = número de CSV REDEXT/REDCOS
Errores de lectura = 0
Filas guardadas ocean_validation REDEXT/REDCOS > 0
hs_missing_pct razonable
Validación final REDEXT/REDCOS superada
```

Si alguna boya no trae coordenadas, se guarda como `CAN_VALIDATION_UNASSIGNED`. Para usarla en comparaciones espaciales exactas, rellena `REDEXT_REDCOS_COORD_FALLBACK` y reejecuta desde la celda 10.